In [ ]:
# %% [code]
# --- Standard ---
import os
import sys
import json
from datetime import datetime
from dateutil.relativedelta import relativedelta

# --- Data/plot ---
import pandas as pd
import matplotlib.pyplot as plt

# --- HTTP ---
import requests

# --- Spark ---
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, DoubleType

# --- Cassandra (Python-driver for å lage keyspace/tabl.) ---
from cassandra.cluster import Cluster
from cassandra.auth import PlainTextAuthProvider

# --- MongoDB ---
from pymongo import MongoClient

# ====== KONFIGURER HER ======
# Cassandra
CASSANDRA_HOST = os.getenv("CASSANDRA_HOST", "127.0.0.1")
CASSANDRA_PORT = int(os.getenv("CASSANDRA_PORT", "9042"))
CASSANDRA_USER = os.getenv("CASSANDRA_USER", "cassandra")
CASSANDRA_PASS = os.getenv("CASSANDRA_PASS", "cassandra")
KEYSPACE = "elhub"
TABLE = "production_2021_by_hour"

# MongoDB
MONGODB_URI = os.getenv("MONGODB_URI", "mongodb://localhost:27017")
MONGO_DB = "elhub"
MONGO_COLLECTION = "production_2021_by_hour"

# Lokalt nett (Spark bind fix for macOS)
LOCAL_IP = "127.0.0.1"

# --- Elhub API (samme mønster som vennen din brukte i HTML) ---
BASE_URL = "https://api.elhub.no/energy-data/v0/price-areas"
DATASET = "PRODUCTION_PER_GROUP_MBA_HOUR"
START_DATE = "2021-01-01"
END_DATE   = "2021-12-31"

# Hvis du trenger proxy/headers (som auth), sett her. Tom som standard.
DEFAULT_HEADERS = {
    # "X-API-Key": "DIN_NØKKEL_HER",
}
DEFAULT_TIMEOUT = (15, 60)  # (connect, read)


In [ ]:
# %% [code]
spark = (
    SparkSession.builder
    .master("local[*]")  # eksplisitt lokal
    .appName("IND320-Elhub-2021")
    # Cassandra connector (Spark 3.5 / Scala 2.12)
    .config("spark.jars.packages", "com.datastax.spark:spark-cassandra-connector_2.12:3.5.1")
    # Cassandra connection
    .config("spark.cassandra.connection.host", CASSANDRA_HOST)
    .config("spark.cassandra.connection.port", str(CASSANDRA_PORT))
    .config("spark.cassandra.auth.username", CASSANDRA_USER)
    .config("spark.cassandra.auth.password", CASSANDRA_PASS)
    .config("spark.cassandra.output.consistency.level", "LOCAL_QUORUM")
    # Tidsone
    .config("spark.sql.session.timeZone", "UTC")
    # Viktig: bind til loopback/IPv4 (fikser 'Service sparkDriver failed after 16 retries')
    .config("spark.driver.bindAddress", LOCAL_IP)
    .config("spark.driver.host", LOCAL_IP)
    .config("spark.local.ip", LOCAL_IP)
    .config("spark.driver.extraJavaOptions", "-Djava.net.preferIPv4Stack=true")
    # Valgfritt: slå av web-UI for å unngå portkrøll
    .config("spark.ui.enabled", "false")
    # Cassandra Spark extensions
    .config("spark.sql.extensions", "com.datastax.spark.connector.CassandraSparkExtensions")
    .getOrCreate()
)
print("✅ Spark er i gang")


In [ ]:
# %% [code]
auth = PlainTextAuthProvider(CASSANDRA_USER, CASSANDRA_PASS)
cluster = Cluster([CASSANDRA_HOST], port=CASSANDRA_PORT, auth_provider=auth)
session = cluster.connect()

session.execute(f"""
CREATE KEYSPACE IF NOT EXISTS {KEYSPACE}
WITH replication = {{ 'class': 'SimpleStrategy', 'replication_factor': '1' }};
""")

session.set_keyspace(KEYSPACE)

session.execute(f"""
CREATE TABLE IF NOT EXISTS {TABLE} (
    pricearea text,
    productiongroup text,
    starttime timestamp,
    quantitykwh double,
    PRIMARY KEY ((pricearea, productiongroup), starttime)
) WITH CLUSTERING ORDER BY (starttime ASC);
""")

print(f"✅ Cassandra klar: {KEYSPACE}.{TABLE}")


In [ ]:
# %% [code]
def fetch_data(area: str, start_iso: str, end_iso: str) -> pd.DataFrame:
    """
    Matcher kompisen din sin HTML:
    GET {BASE_URL}/{area}?dataset=PRODUCTION_PER_GROUP_MBA_HOUR&startDate=...&endDate=...
    """
    url = f"{BASE_URL}/{area}"
    params = {
        "dataset": DATASET,
        "startDate": start_iso,
        "endDate": end_iso,
    }
    try:
        r = requests.get(url, params=params, headers=DEFAULT_HEADERS, timeout=DEFAULT_TIMEOUT)
        # Debug-utskrift for å se hva API svarer
        print(f"HTTP {r.status_code} {r.url}")
        if r.status_code != 200:
            # Vis litt responsinnhold ved feil
            txt = r.text[:300].replace("\n", " ")
            print(f"  Body (trunc): {txt}")
            return pd.DataFrame()

        payload = r.json()
        lst = payload.get("productionPerGroupMbaHour", [])
        if not lst:
            print("  (tom liste: productionPerGroupMbaHour)")
            return pd.DataFrame()

        df = pd.json_normalize(lst)
        # Forventede felter -> flate/standardiser
        # Eksempel på typiske felt: priceArea, productionGroup, startTime, quantityKwh
        rename = {
            "priceArea": "pricearea",
            "productionGroup": "productiongroup",
            "startTime": "starttime",
            "quantityKwh": "quantitykwh",
        }
        df = df.rename(columns=rename)
        # Typer
        if "starttime" in df.columns:
            df["starttime"] = pd.to_datetime(df["starttime"], utc=True)
        if "quantitykwh" in df.columns:
            df["quantitykwh"] = pd.to_numeric(df["quantitykwh"], errors="coerce")
        return df[["pricearea", "productiongroup", "starttime", "quantitykwh"]].dropna()
    except requests.RequestException as e:
        print("⚠️ HTTP-feil:", e)
        return pd.DataFrame()


In [ ]:
# %% [code]
areas_resp = requests.get(BASE_URL, headers=DEFAULT_HEADERS, timeout=DEFAULT_TIMEOUT)
if areas_resp.status_code == 200:
    areas = areas_resp.json().get("priceAreas", ["NO1", "NO2", "NO3", "NO4", "NO5"])
else:
    print("⚠️ Klarte ikke hente areas – bruker NO1–NO5 som default")
    areas = ["NO1", "NO2", "NO3", "NO4", "NO5"]

print("📍 Price areas:", areas)

# NB: Kompisen din brukte fast +02:00 i hele året i HTML-en sin.
# Vi reproduserer det for å matche oppførselen 1:1.
start_dt = datetime.fromisoformat(START_DATE + "T00:00:00+02:00")
end_dt   = datetime.fromisoformat(END_DATE   + "T23:59:59+02:00")

all_parts = []

for area in areas:
    print(f"⬇️ Fetching {area} ... ")
    current = start_dt
    got_any = False
    while current < end_dt:
        nxt = current + relativedelta(months=1)
        if nxt > end_dt:
            nxt = end_dt

        start_iso = current.isoformat()
        end_iso   = nxt.isoformat()

        df = fetch_data(area, start_iso, end_iso)
        if not df.empty:
            all_parts.append(df)
            got_any = True

        current = nxt

    if not got_any:
        print("   no data")

if not all_parts:
    raise RuntimeError("No data fetched from API; cannot proceed.")
    
pdf = pd.concat(all_parts, ignore_index=True)
print("✅ Rows fetched:", len(pdf))
pdf.head()


In [ ]:
# %% [code]
schema = StructType([
    StructField("pricearea", StringType(), False),
    StructField("productiongroup", StringType(), False),
    StructField("starttime", TimestampType(), False),
    StructField("quantitykwh", DoubleType(), True),
])

sdf = spark.createDataFrame(pdf, schema=schema)

(sdf.write
    .format("org.apache.spark.sql.cassandra")
    .mode("append")
    .options(table=TABLE, keyspace=KEYSPACE)
    .save())

print("✅ Skrevet til Cassandra")


In [ ]:
# %% [code]
read_back = (
    spark.read
    .format("org.apache.spark.sql.cassandra")
    .options(table=TABLE, keyspace=KEYSPACE)
    .load()
)

print("✅ Antall rader i Cassandra:", read_back.count())
read_back.limit(5).toPandas()


In [ ]:
# %% [code]
client = MongoClient(MONGODB_URI)
mdb = client[MONGO_DB]
coll = mdb[MONGO_COLLECTION]

# (Valgfritt) Tøm først for ren kjøring
coll.delete_many({})

# Sett inn i batches for ikke å kvele minnet
BATCH = 50_000
rows = pdf.to_dict(orient="records")

for i in range(0, len(rows), BATCH):
    chunk = rows[i:i+BATCH]
    coll.insert_many(chunk)

print("✅ MongoDB insert ferdig")
print("Docs i Mongo:", coll.count_documents({}))


In [ ]:
# %% [code]
client = MongoClient(MONGODB_URI)
mdb = client[MONGO_DB]
coll = mdb[MONGO_COLLECTION]

# (Valgfritt) Tøm først for ren kjøring
coll.delete_many({})

# Sett inn i batches for ikke å kvele minnet
BATCH = 50_000
rows = pdf.to_dict(orient="records")

for i in range(0, len(rows), BATCH):
    chunk = rows[i:i+BATCH]
    coll.insert_many(chunk)

print("✅ MongoDB insert ferdig")
print("Docs i Mongo:", coll.count_documents({}))


In [ ]:
# %% [code]
# Velg område
chosen = "NO1"

df_area = pdf[pdf["pricearea"] == chosen].copy()
df_area["date"] = pd.to_datetime(df_area["starttime"]).dt.tz_convert("Europe/Oslo")

# --- Pie: total 2021 per produksjonsgruppe ---
sum_grp = df_area.groupby("productiongroup", as_index=False)["quantitykwh"].sum()

plt.figure()
plt.pie(sum_grp["quantitykwh"], labels=sum_grp["productiongroup"], autopct="%1.1f%%")
plt.title(f"{chosen} — Total production 2021 (kWh)")
plt.show()

# --- Line: første måned (januar) med separate linjer per gruppe ---
jan_mask = (df_area["date"] >= "2021-01-01") & (df_area["date"] < "2021-02-01")
df_jan = df_area[jan_mask].copy()

if df_jan.empty:
    print("⚠️ Ingen data i januar for", chosen)
else:
    pivot = df_jan.pivot_table(index="date", columns="productiongroup", values="quantitykwh", aggfunc="sum").sort_index()
    plt.figure()
    pivot.plot()
    plt.title(f"{chosen} — Hourly production, January 2021")
    plt.xlabel("Time")
    plt.ylabel("kWh")
    plt.legend(title="Production Group", bbox_to_anchor=(1.05, 1), loc="upper left")
    plt.show()


In [ ]:
# %% [code]
# Test én enkel måned/område og print litt mer av responsen:
test_area = "NO1"
test_start = "2021-01-01T00:00:00+02:00"
test_end   = "2021-01-31T23:59:59+02:00"

url = f"{BASE_URL}/{test_area}"
params = {"dataset": DATASET, "startDate": test_start, "endDate": test_end}
try:
    r = requests.get(url, params=params, headers=DEFAULT_HEADERS, timeout=DEFAULT_TIMEOUT)
    print("URL:", r.url)
    print("Status:", r.status_code)
    print("Headers:", dict(r.headers))
    body = r.text
    print("Body (first 500 chars):", body[:500].replace("\n", " "))
except Exception as e:
    print("HTTP error:", e)

# Hvis body ikke har "productionPerGroupMbaHour", er årsaken upstream.
# Da er det korrekt at Cassandra/Mongo er tomme – DBene er ikke problemet i seg selv.
